In [2]:
import duckdb
from pathlib import Path

# Path to the DuckDB database file
DATA_DIR = Path("../data/raw")

# Connect to the DuckDB database
con = duckdb.connect("../northwind.duckdb")

# Task 04: Revenue Foundation

## Goal

Calculate revenue correctly from `order_details`.

## What you will learn

```text
gross revenue
discount amount
net revenue
line-level calculation
order-level revenue
SQL views
```

## Main formulas

```text
gross_revenue = unitPrice * quantity
discount_amount = unitPrice * quantity * discount
net_revenue = unitPrice * quantity * (1 - discount)
```

## Step 1: Inspect revenue inputs

```python
display(con.sql("""
    SELECT COUNT(*) AS total_rows,
           MIN(unitPrice) AS min_unit_price,
           MAX(unitPrice) AS max_unit_price,
           MIN(quantity) AS min_quantity,
           MAX(quantity) AS max_quantity,
           MIN(discount) AS min_discount,
           MAX(discount) AS max_discount
    FROM order_details;
""").df())
```

In [8]:
display(con.sql("""
    SELECT COUNT(*) AS total_rows,
           MIN(unitPrice) AS min_unit_price,
           MAX(unitPrice) AS max_unit_price,
           MIN(quantity) AS min_quantity,
           MAX(quantity) AS max_quantity,
           MIN(discount) AS min_discount,
           MAX(discount) AS max_discount
    FROM order_details;
""").df())

,total_rows,min_unit_price,max_unit_price,min_quantity,max_quantity,min_discount,max_discount
0,2155,2.0,263.5,1,130,0.0,0.25


## Step 2: Check bad revenue inputs

```python
display(con.sql("""
    SELECT *
    FROM order_details
    WHERE unitPrice IS NULL
       OR quantity IS NULL
       OR discount IS NULL
       OR unitPrice <= 0
       OR quantity <= 0
       OR discount < 0
       OR discount > 1;
""").df())
```

In [11]:
con.sql("""
SELECT COUNT(*) AS Bad_rows 
FROM order_details
WHERE unitPrice <=0 OR quantity<=0 OR discount<0 OR discount >1 OR unitPrice IS NULL OR quantity IS NULL OR discount IS NULL;        

""").df()

,Bad_rows
0,0


Empty result means revenue inputs look safe.

## Step 3: Calculate gross revenue

```python
display(con.sql("""
    SELECT orderID, productID, unitPrice, quantity, discount,
           unitPrice * quantity AS gross_revenue
    FROM order_details
    LIMIT 10;
""").df())
```


## Step 4: Calculate discount amount

```python
display(con.sql("""
    SELECT orderID, productID, unitPrice, quantity, discount,
           unitPrice * quantity AS gross_revenue,
           unitPrice * quantity * discount AS discount_amount
    FROM order_details
    LIMIT 10;
""").df())
```

## Step 5: Calculate net revenue

```python
display(con.sql("""
    SELECT orderID, productID, unitPrice, quantity, discount,
           unitPrice * quantity AS gross_revenue,
           unitPrice * quantity * discount AS discount_amount,
           unitPrice * quantity * (1 - discount) AS net_revenue
    FROM order_details
    LIMIT 10;
""").df())
```

## Step 6: Total net revenue

```python
display(con.sql("""
    SELECT ROUND(SUM(unitPrice * quantity * (1 - discount)), 2) AS total_net_revenue
    FROM order_details;
""").df())
```

## Step 7: Revenue per order

```python
display(con.sql("""
    SELECT orderID,
           ROUND(SUM(unitPrice * quantity * (1 - discount)), 2) AS order_revenue,
           COUNT(productID) AS product_lines,
           SUM(quantity) AS total_units
    FROM order_details
    GROUP BY orderID
    ORDER BY order_revenue DESC;
""").df())
```

## Step 8: Create sales_lines view

```python
con.sql("""
CREATE OR REPLACE VIEW sales_lines AS
SELECT od.orderID,
       od.productID,
       od.unitPrice,
       od.quantity,
       od.discount,
       od.unitPrice * od.quantity AS gross_revenue,
       od.unitPrice * od.quantity * od.discount AS discount_amount,
       od.unitPrice * od.quantity * (1 - od.discount) AS net_revenue
FROM order_details AS od;
""")
```

Test it:

```python
display(con.sql("SELECT * FROM sales_lines LIMIT 10;").df())
```

## Step 9: Validate sales_lines

```python
display(con.sql("""
    SELECT COUNT(*) AS sales_line_rows,
           ROUND(SUM(net_revenue), 2) AS total_net_revenue
    FROM sales_lines;
""").df())
```

```python
display(con.sql("SELECT COUNT(*) AS order_detail_rows FROM order_details;").df())
```

## Observation template

```text
Revenue input summary:
Bad revenue inputs found:
Total net revenue:
Highest revenue order:
Difference between product_lines and total_units:
Was sales_lines view created:
Why sales_lines is useful:
```

## Stop point

You are done when you understand that revenue comes from `order_details`, must include `quantity`, and must include `discount`.